Tải các thư viện liên quan

In [ ]:
%pip install paddlex paddlepaddle paddleocr

1. Mô hình chạy trên local

In [ ]:
from paddleocr import PaddleOCR
import time

start_time = time.perf_counter()

ocr = PaddleOCR(
    ocr_version="PP-OCRv5",
    # text_detection_model_name="PP-OCRv4_mobile_det",
    # text_recognition_model_name="PP-OCRv4_mobile_rec",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    device='cpu',
    enable_mkldnn=False,
    lang="en"
)

result = ocr.predict("203110926212582971001.jpg")

end_time = time.perf_counter()
print(f"⏱️ Thời gian quét ảnh thực tế: {end_time - start_time:.3f} giây\n")

for res in result:
    res.save_to_img("output.png")
    res.print()
    if res and 'rec_texts' in res:
        for text in res['rec_texts']:
            print(text)

2. Mô hình chạy trên API Key của BanduStudio

In [32]:
# Please make sure the requests library is installed
# pip install requests
import json
import os
import requests
import sys
import time

JOB_URL = "https://paddleocr.aistudio-app.com/api/v2/ocr/jobs"
TOKEN = "1d4a9cc70271f482b22714a41e440cccc26f2483"
MODEL = "PP-OCRv6"

file_path = "203110926212582971001.jpg"

headers = {
    "Authorization": f"bearer {TOKEN}",
}

optional_payload = {
    "useDocOrientationClassify": False,
    "useDocUnwarping": False,
    "useTextlineOrientation": False,
}

print(f"Processing file: {file_path}")

if file_path.startswith("http"):
    # URL Mode
    headers["Content-Type"] = "application/json"
    payload = {
        "fileUrl": file_path,
        "model": MODEL,
        "optionalPayload": optional_payload
    }
    job_response = requests.post(JOB_URL, json=payload, headers=headers)
else:
    # Local File Mode
    if not os.path.exists(file_path):
        print(f"Error: File not found at {file_path}")
        sys.exit(1)
        
    data = {
        "model": MODEL,
        "optionalPayload": json.dumps(optional_payload)
    }
    
    with open(file_path, "rb") as f:
        files = {"file": f}
        job_response = requests.post(JOB_URL, headers=headers, data=data, files=files)

print(f"Response status: {job_response.status_code}")

if job_response.status_code != 200:
    print(f"Response content: {job_response.text}")

assert job_response.status_code == 200
jobId = job_response.json()["data"]["jobId"]
print(f"Job submitted successfully. job id: {jobId}")
print("Start polling for results")

jsonl_url = ""
while True:
    job_result_response = requests.get(f"{JOB_URL}/{jobId}", headers=headers)
    assert job_result_response.status_code == 200
    state = job_result_response.json()["data"]["state"]
    if state == 'pending':
        print("The current status of the job is pending")
    elif state == 'running':
        try:
            total_pages = job_result_response.json()['data']['extractProgress']['totalPages']
            extracted_pages = job_result_response.json()['data']['extractProgress']['extractedPages']
            print(f"The current status of the job is running, total pages: {total_pages}, extracted pages: {extracted_pages}")
        except KeyError:
             print("The current status of the job is running...")
    elif state == 'done':
        extracted_pages = job_result_response.json()['data']['extractProgress']['extractedPages']
        start_time = job_result_response.json()['data']['extractProgress']['startTime']
        end_time = job_result_response.json()['data']['extractProgress']['endTime']
        print(f"Job completed, successfully extracted pages: {extracted_pages}, start time: {start_time}, end time: {end_time}")
        jsonl_url = job_result_response.json()['data']['resultUrl']['jsonUrl']
        break
    elif state == "failed":
        error_msg = job_result_response.json()['data']['errorMsg']
        print(f"Job failed, failure reason：{error_msg}")
        sys.exit()

    time.sleep(5)

if jsonl_url:
    jsonl_response = requests.get(jsonl_url)
    jsonl_response.raise_for_status()
    lines = jsonl_response.text.strip().split('\n')
    output_dir = "output"
    os.makedirs(output_dir, exist_ok=True)
    
    text_file_path = os.path.join(output_dir, "recognized_text.txt")
    
    with open(text_file_path, "w", encoding="utf-8") as f_txt:
        page_num = 0
        for line_num, line in enumerate(lines, start=1):
            line = line.strip()
            if not line:
                continue
            
            result = json.loads(line)["result"]
            
            for i, res in enumerate(result["ocrResults"]):
                print(f"\n--- NỘI DUNG TRANG {page_num + 1} ---")
                
                # Trích xuất danh sách text từ prunedResult
                rec_texts = res.get("prunedResult", {}).get("rec_texts", [])
                
                for text in rec_texts:
                    print(text)
                    f_txt.write(text + "\n")
                
                # Tải và lưu ảnh có bounding box
                image_url = res.get("ocrImage")
                if image_url:
                    img_response = requests.get(image_url)
                    if img_response.status_code == 200:
                        filename = f"output/img_output_{page_num}.jpg"
                        with open(filename, "wb") as f:
                            f.write(img_response.content)
                        print(f"\nLưu thành công : {filename}")
                    else:
                        print(f"\nLưu thất bại : {img_response.status_code}")
                
                page_num += 1

    print(f"\n✅ Đã lưu toàn bộ văn bản vào: {text_file_path}")

Processing file: 203110926212582971001.jpg
Response status: 200
Job submitted successfully. job id: 84256655991218176
Start polling for results
Job completed, successfully extracted pages: 1, start time: 2026-08-21 20:05:52, end time: 2026-08-21 20:05:52

--- NỘI DUNG TRANG 1 ---
COLLOCATIONS -2
Choose A, B, C or D to indicate the correct answer to each of the following questions.
1. Her ideas have
a lot of attention in the scientific community.
A. attracted
B. attained
C. caught
D. caused
2. Before doing the
, farmers have to pump the water into the field.
A. ploughing
B. transplanting
C. harrowing
D. harvesting
3. She finally achieved her
of visiting the USA.
A. objective
B. target
C. desires
D. ambition
4. When I was reading a book in my room last night, I heard the sound of
glass.
A. breaking
B. slipping
C. dropping
D. bursting
5. If you
your mind about attending Mr. Jones's lecturer, just give me a call.
A. change
B. keep
C. decide
D. give
6. After a serious accident last month, t